# Interactive Script: **Cloud and Shadow Masking**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
Two cloud detection routes, how to apply either one to a data cube, and how to add cloud shadows.

## Contents

1. [Two Routes to a Cloud Mask](#1-two-routes-to-a-cloud-mask)
2. [SCL Masking in the Builder](#2-scl-masking-in-the-builder)
3. [The SCL Mask of an Existing Cube](#3-the-scl-mask-of-an-existing-cube)
4. [s2cloudless: Cloud Probability Cube](#4-s2cloudless-cloud-probability-cube)
5. [Binary Masks from the Probability Map](#5-binary-masks-from-the-probability-map)
6. [Apply a Mask to the Data Cube](#6-apply-a-mask-to-the-data-cube)
7. [One-Call Workflow](#7-one-call-workflow)
8. [Filter by Cloud Percentage](#8-filter-by-cloud-percentage)
9. [Update Existing Cloud Layers](#9-update-existing-cloud-layers)
10. [Cloud Shadows](#10-cloud-shadows)

---

In [ ]:
from stac2cube import (
    get_stac_layers,
    get_cloud_layers,
    mask_stac_clouds,
    mask_from_probability,
    cloud_filter,
    open_cube,
    export_stac,
)
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

## 0. Setup

The notebook builds its own base cube so it can be run on its own.

s2cloudless masking re-queries the archive for the matching Sentinel-2 L1C scenes and puts them on the cube's own grid, which is read off the cube rather than re-derived from the area. So the mask lines up pixel for pixel whether the cube was built from a polygon file or from a bounding box, and a clipped or reprojected cube can be masked too.

In [ ]:
polygon = "../polygons/test.gpkg"

BASE = "../results/test_cloudbase.nc"          # the cube this notebook masks
BASE_MASKED_70 = "../results/test_cloudbase_masked_70.nc"

get_stac_layers(
    mission="s2",
    polygon=polygon,
    resolution=10,
    daterange=["2024-04-01", "2024-04-10"],
    bands=["blue", "green", "red", "nir"],
    indices=["ndvi", "ndwi"],
    max_cc=100,
    output=BASE,
    q=True,
)

with open_cube(BASE) as ds:
    print(ds["Time_Series"].shape, ds["Time_Series"].time.values)

## 1. Two Routes to a Cloud Mask

| | **SCL** | **s2cloudless** |
|---|---|---|
| where it comes from | the Scene Classification Layer shipped with every L2A scene | a LightGBM model run on the matching L1C scene |
| what you get | one fixed classification (classes 8, 9, 10 = cloud) | a continuous cloud probability per pixel, 0 to 100 |
| threshold | none, the classes are fixed | yours, any value, and several at once |
| cost | free, the layer is already there | downloads the L1C scenes and runs the model, so it takes time |
| where | `cloud_masking=True` in `get_stac_layers` | `get_cloud_layers` in this notebook |

Rule of thumb: SCL for a quick or very large job, s2cloudless when the mask quality matters, for example before co-registration or when you count cloud-free observations.

Every cube records which route produced it in the `cloud_status` attribute:

| `cloud_status` | meaning |
|---|---|
| `clouds_not_detected` | no detection ran |
| `clouds_detected` | clouds were detected but the pixels were kept |
| `scl_masked` | SCL clouds set to no-data |
| `scl_shadow_masked` | SCL clouds and shadows set to no-data |
| `cloud_mask_<N>` | s2cloudless at threshold N |

The update, masking and co-registration tools read this attribute, which is how an updated cube gets exactly the same treatment as the dates already in it.

## 2. SCL Masking in the Builder

`cloud_masking=True` sets cloudy pixels to no-data while the cube is built.

`keep_clouds=True` runs the same detection but keeps the pixels. You still get the `cloud_percentage` time coordinate, so you can filter or sort scenes without losing any imagery. This is what you want for natural-looking animations, or when you plan to mask later with your own threshold.

In [ ]:
stac_masked = get_stac_layers(
    mission="s2",
    polygon=polygon,
    resolution=10,
    daterange=["2024-04-01", "2024-04-20"],
    bands=["blue", "green", "red", "nir"],
    indices=["ndvi"],
    max_cc=100,
    cloud_masking=True,
    output="../results/test_scl_masked.nc",
    q=True,
)
print(stac_masked.attrs["cloud_status"])
print(stac_masked.cloud_percentage.values)

### 2.1 Keeping the binary mask

`cloud_mask_output` writes the binary mask itself (1 = cloud, 0 = clear) as a separate cube, and `return_cloud_mask=True` returns it in memory alongside the cube. Useful when you want to co-register a cube that keeps its clouds (notebook 3), or to compare masks.

In [ ]:
cube_kept, mask = get_stac_layers(
    mission="s2",
    polygon=polygon,
    resolution=10,
    daterange=["2024-04-01", "2024-04-20"],
    bands=["blue", "green", "red", "nir"],
    max_cc=100,
    cloud_masking=True,
    keep_clouds=True,                                  # detect, but do not remove
    cloud_mask_output="../results/test_scl_mask.nc",   # write the binary mask
    return_cloud_mask=True,                            # and hand it back
    output=None,
    q=True,
)
print(cube_kept.attrs["cloud_status"])
print(mask.band.values, mask.shape)

### 2.2 Percentage of the area covered by cloud

`cloud_percentage` is a time coordinate on every cube where detection ran. It is the share of **your area** under cloud, not the cloud cover of the whole Sentinel-2 tile that `max_cc` filters on.

In [ ]:
for t, c in zip(cube_kept.time.values, cube_kept.cloud_percentage.values):
    print(f"{str(t)[:10]}   {int(c):3d} %")

## 3. The SCL Mask of an Existing Cube

If you already have a cube that was built with SCL detection but did not keep its mask, the mask can be reconstructed. It re-queries the catalogue with the cube's own stored parameters and keeps only the scenes the cube holds, so it lines up date for date.

Needs internet, and only covers the dates already in the cube: a cube that was narrowed afterwards cannot get its dropped dates back this way.

In [ ]:
from stac2cube import build_cloud_mask_cube

scl_mask = build_cloud_mask_cube(
    "../results/test_scl_masked.nc",
    output="../results/test_scl_mask_rebuilt.nc",
)
scl_mask

The rebuilt mask carries the metadata needed to extend it later, so new dates can be added without recomputing the old ones.

In [ ]:
from stac2cube import update_cloud_mask_cube

scl_mask_extended = update_cloud_mask_cube(
    "../results/test_scl_mask_rebuilt.nc",
    daterange=["2024-04-01", "2024-04-30"],
    output=None,
)
print(scl_mask_extended.time.values)

## 4. s2cloudless: Cloud Probability Cube

`get_cloud_layers` downloads the Sentinel-2 **L1C** scenes matching your area and runs the s2cloudless model on them. The result is a `Cloud_Stack` cube with a `cloud_prob` band (0 to 100), plus one binary `cloud_mask_<N>` band per threshold you ask for.

This step cannot stay lazy: the model needs the pixels in memory. On a long time series, run it as an HPC job and read the exported file back.

### 4.1 From an area and a date range

In [ ]:
cloud_stac = get_cloud_layers(
    polygon=polygon,
    daterange=["2024-04-01", "2024-04-10"],
    threshold=[40, 50, 60, 70, 80, 90],   # percent; None returns the probability only
    clip_raster=False,
    output_clouds="../results/test_cloud.nc",
)
cloud_stac

### 4.2 From an existing cube

Better, in almost every case: `input_cube` takes the area, the projection, the resolution and the **exact acquisition dates** from a cube you already have, so the mask aligns to it pixel for pixel. Without it the probability is computed over a continuous date range, which for a seasonal cube means dates the cube does not contain.

`masking` does the same and additionally masks that cube at the end (chapter 7).

In [ ]:
cloud_stac = get_cloud_layers(
    input_cube=BASE,
    threshold=[40, 70],
    output_clouds="../results/test_cloud.nc",
)
print(cloud_stac.band.values)
print(cloud_stac.time.values)

> **Dates without an L1C scene.** element84 is the only catalogue serving a usable L1C archive, and it is sparse before roughly 2021. A cube built from Planetary Computer or terrabyte can therefore hold dates with no L1C scene to run the model on. `missing_l1c` decides what happens then:
> - `"scl"` (default) fills those dates with the SCL binary mask from the cube's own source, so the cube ends up fully masked. Such a cube is a hybrid and cannot be updated in the builder.
> - `"drop"` leaves those dates out of the `Cloud_Stack`, which drops them from the masked cube too.
> - `"error"` stops instead.

### 4.3 Read an exported cloud cube back

In [ ]:
with open_cube("../results/test_cloud.nc") as ds:
    cloud_stac = ds["Cloud_Stack"].load()

cloud_stac.band.values

### 4.4 Look at the layers

In [ ]:
date = str(cloud_stac.time.values[0])[:10]

cloud_probability = cloud_stac.sel(band="cloud_prob")
cloud_mask_70 = cloud_stac.sel(band="cloud_mask_70")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(cloud_probability.sel(time=date), cmap="turbo", vmin=0, vmax=100)
axes[0].set_title(f"Cloud probability, {date}")
axes[1].imshow(cloud_mask_70.sel(time=date), cmap="gray")
axes[1].set_title(f"Binary mask at 70%, {date}")
for ax in axes:
    ax.axis("off")
plt.show()

### 4.5 Compare against the imagery

The overlay viewer puts the cube next to its cloud layer with an adjustable opacity, which is the quickest way to judge whether a threshold is too strict or too loose.

In [ ]:
from stac2cube import interactive_cloud_overlay_view

with open_cube(BASE) as ds:
    stac = ds["Time_Series"].load()

interactive_cloud_overlay_view(spectral=stac, cloud=cloud_stac)

## 5. Binary Masks from the Probability Map

If you exported the probability without thresholds, or want to try more of them, derive the masks from the stored `cloud_prob` band. No download, no model run.

`average_over` and `dilation_size` are the s2cloudless post-processing, in pixels: a disk-mean convolution followed by a dilation. Larger values give smoother, more generous cloud blobs. Defaults are 4 and 2.

In [ ]:
cloud_probability = cloud_stac.sel(band="cloud_prob")

mask_stac = mask_from_probability(
    cloud_probability=cloud_probability,
    threshold=[40, 60, 70, 90],
)
mask_stac.band.values

In [ ]:
date = str(mask_stac.time.values[0])[:10]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, thr in zip(axes, [40, 60, 70, 90]):
    ax.imshow(mask_stac.sel(time=date, band=f"cloud_mask_{thr}"), cmap="gray")
    ax.set_title(f"threshold {thr}%")
    ax.axis("off")
plt.suptitle(f"Cloud masks, {date}")
plt.show()

**Put the new masks back into the cloud cube and export**

In [ ]:
new_bands = set(map(str, mask_stac["band"].values))
base_bands = [b for b in map(str, cloud_stac["band"].values) if b not in new_bands]
base = cloud_stac.sel(band=base_bands)

combined = xr.concat([base, mask_stac], dim="band").transpose("time", "band", "y", "x")
combined.name = "Cloud_Stack"
combined.attrs = cloud_stac.attrs

export_stac(combined, "../results/test_cloud.nc", overwrite=True, var_name="Cloud_Stack")

## 6. Apply a Mask to the Data Cube

`mask_stac_clouds` sets the pixels flagged by one mask band to no-data. One threshold per call: pick the band, get the masked cube.

In [ ]:
with open_cube(BASE) as ds:
    stac = ds["Time_Series"].load()

with open_cube("../results/test_cloud.nc") as ds:
    cloud = ds["Cloud_Stack"].load()

In [ ]:
# The date axes must match. This shows any mismatch before the call fails.
t_cube = set(np.asarray(stac.time.values).astype("datetime64[D]"))
t_cloud = set(np.asarray(cloud.time.values).astype("datetime64[D]"))
print("in cube but not in cloud cube:", sorted(t_cube - t_cloud))
print("in cloud cube but not in cube:", sorted(t_cloud - t_cube))

In [ ]:
masked_stac = mask_stac_clouds(
    stac=stac,
    cloud=cloud,
    mask_layer="cloud_mask_70",
    output="../results/test_masked.nc",
)

# File paths work too:
# mask_stac_clouds(BASE, "../results/test_cloud.nc", "cloud_mask_70",
#                  "../results/test_masked.nc")

In [ ]:
with open_cube("../results/test_masked.nc") as ds:
    masked_stac = ds["Time_Series"].load()

print(masked_stac.attrs["cloud_status"])
masked_stac

In [ ]:
date = str(masked_stac.time.values[0])[:10]

def stretch(arr, p_low=2, p_high=98):
    lo, hi = np.nanpercentile(arr, [p_low, p_high])
    return np.clip((arr - lo) / (hi - lo), 0, 1)

before = stac.sel(time=date, band=["red", "green", "blue"]).values.transpose(1, 2, 0)
after = masked_stac.sel(time=date, band=["red", "green", "blue"]).values.transpose(1, 2, 0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(stretch(before))
axes[0].set_title(f"Original, {date}")
axes[1].imshow(stretch(after))
axes[1].set_title(f"Masked at 70%, {date}")
for ax in axes:
    ax.axis("off")
plt.show()

> **Why keep `max_cc` at 100.** `max_cc` filters on the cloud cover of the whole Sentinel-2 tile, which can be over 100 km across. A tile reported at 60% cloud can be perfectly clear over a small area inside it. Filtering there throws away usable scenes before you ever see them. Keep `max_cc=100`, then filter on `cloud_percentage`, which is measured over your area (chapter 8).

## 7. One-Call Workflow

`masking=<cube>` runs the whole chain: take the dates and grid from the cube, compute the probability, threshold it, mask the cube, export both.

`output_masked=None` writes next to the input with a `_masked_<threshold>` suffix. `output_clouds=None` means the probability layers are not kept.

In [ ]:
get_cloud_layers(
    masking=BASE,              # the cube to mask
    threshold=70,                              # one value here, not a list
    output_clouds="../results/test_cloud.nc",  # None = do not keep the probability
    output_masked=None,                        # None = <input>_masked_70.nc
)

In [ ]:
with open_cube(BASE_MASKED_70) as ds:
    print(ds["Time_Series"].attrs["cloud_status"])
    print(ds["Time_Series"].cloud_percentage.values)

## 8. Filter by Cloud Percentage

`cloud_filter` keeps the scenes at or below a cloud percentage. It takes a path, a Dataset or a DataArray.

In [ ]:
with open_cube(BASE_MASKED_70) as ds:
    masked_stac = ds["Time_Series"].load()

print("before:", masked_stac.cloud_percentage.values)

masked_filtered = cloud_filter(masked_stac, max_cloud=20)
print("after :", masked_filtered.cloud_percentage.values)

In [ ]:
export_stac(masked_filtered, "../results/test_cloud_filtered.nc")

> The same filter is available while building, as `scene_cloud_coverage=20` in `get_stac_layers`, which never puts the cloudy scenes in the cube in the first place. See notebook 1, chapter 3.

## 9. Update Existing Cloud Layers

Extending a cloud cube works like extending a data cube: only the missing dates are computed, the existing ones are never recomputed.

> If there is nothing new to compute, the call raises `ValueError: The probability map is up to date. Nothing to update!` rather than returning the cube unchanged. Check the dates first if you are not sure.

### 9.1 Cloud layers only

Widen the date range and the new scenes are added to the cloud cube. `output=None` returns the extended stack without touching the file; give it a path to write it.

In [ ]:
with open_cube("../results/test_cloud.nc") as ds:
    print(ds["Cloud_Stack"].time.values)

In [ ]:
cloud_updated = get_cloud_layers(
    update="../results/test_cloud.nc",
    daterange=["2024-04-01", "2024-04-20"],
    threshold=[40, 70],
    output=None,                       # a path exports the extended cloud cube
)
cloud_updated.time.values

### 9.2 Cloud layers and the masked cube together

The usual case: the data cube has grown and its mask has to catch up. Extend the data cube first, then run `get_cloud_layers` with both `update` (the cloud cube) and `masking` (the data cube). The new dates are computed, thresholded and applied, and the masked cube is rewritten.

In [ ]:
# Extend the data cube first, so the cloud cube has something to catch up with.
get_stac_layers(
    update=BASE,
    daterange=["2024-04-01", "2024-04-20"],
    output=BASE,
    q=True,
)

with open_cube(BASE) as ds:
    print("data cube :", ds["Time_Series"].time.size, "dates")
with open_cube("../results/test_cloud.nc") as ds:
    print("cloud cube:", ds["Cloud_Stack"].time.size, "dates")

In [ ]:
get_cloud_layers(
    update="../results/test_cloud.nc",
    masking=BASE,
    threshold=70,
)

In [ ]:
with open_cube(BASE_MASKED_70) as ds:
    print(ds["Time_Series"].time.values)

## 10. Cloud Shadows

Shadows are projected **from** the detected clouds along the anti-solar direction, using the per-scene mean solar azimuth from the scene metadata. A pixel inside that projection which is dark in the NIR and is not water is flagged as shadow. This follows the Google Earth Engine s2cloudless tutorial.

Because the shadows come from the clouds, shadow detection always needs cloud detection and the `nir` band.

### 10.1 In the builder, with SCL

`shadow_masking=True` adds the shadow step to an SCL-masked build. Sentinel-2 L2A only, and not available in update mode or with composites.

In [ ]:
stac_shadow = get_stac_layers(
    mission="s2",
    polygon=polygon,
    resolution=10,
    daterange=["2024-04-01", "2024-04-10"],
    bands=["blue", "green", "red", "nir"],
    max_cc=100,
    cloud_masking=True,
    shadow_masking=True,
    nir_dark_threshold=0.18,     # NIR reflectance below which a pixel is dark
    shadow_proj_distance=1.0,    # how far to project, in km
    output="../results/test_shadow_masked.nc",
    q=True,
)
print(stac_shadow.attrs["cloud_status"])

### 10.2 Standalone, on any Sentinel-2 L2A cube

`get_shadow_layers` returns a `Cloud_Stack` with three bands: `cloud_mask`, `shadow_mask` and `cloudshadow_mask` (1 = flagged), plus a `sun_azimuth` time coordinate.

`cloud_source` decides where the clouds come from when you do not pass an existing cloud cube: `"s2cloudless"` downloads the L1C scenes and runs the model, `"scl"` uses SCL classes 8, 9 and 10 and needs no download. If the cube contains the `scl` band, that layer is reused and nothing is downloaded at all.

In [ ]:
from stac2cube import get_shadow_layers

shadow_stack, cube_shadowmasked = get_shadow_layers(
    input_cube=BASE,
    cloud_source="scl",          # or "s2cloudless"
    nir_dark_threshold=0.18,
    proj_distance=1.0,
    masking=True,                # also return the cube with clouds and shadows removed
    output_shadows="../results/test_shadows.nc",
    output_masked="../results/test_cloudshadow_masked.nc",
)
shadow_stack.band.values

In [ ]:
date = str(shadow_stack.time.values[0])[:10]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, b in zip(axes, ["cloud_mask", "shadow_mask", "cloudshadow_mask"]):
    ax.imshow(shadow_stack.sel(time=date, band=b), cmap="gray")
    ax.set_title(b)
    ax.axis("off")
plt.suptitle(f"Cloud and shadow, {date}")
plt.show()

> `proj_distance` is the maximum cloud-to-shadow projection distance in km. Larger values catch the shadows of higher clouds but also flag more genuinely dark surfaces. On an urban test scene, 3 km roughly halved the share of correctly flagged pixels in the mask, so raise it deliberately, not by default.

### 10.3 Adding shadows to an existing s2cloudless cloud cube

If you already ran s2cloudless, the shadows can be derived from one of its binary mask bands and appended to the same `Cloud_Stack`. The new bands are named after that mask, so `cloud_mask_70` gives `shadow_mask_70` and `cloudshadow_mask_70`.

The cloud stack must cover every date of the cube; a stack that is behind the cube is refused with a list of the missing dates. The cube grew in chapter 9, so refresh the stack for its current dates first.

In [ ]:
cloud_now = get_cloud_layers(
    input_cube=BASE,
    threshold=[70],
    output_clouds="../results/test_cloud_now.nc",
)
print(cloud_now.time.size, "dates,", cloud_now.band.values)

In [ ]:
from stac2cube import add_shadow_masks_to_cloud_stack

combined = add_shadow_masks_to_cloud_stack(
    input_cube=BASE,
    cloud="../results/test_cloud_now.nc",
    mask_band="cloud_mask_70",
    output="../results/test_cloud_shadows.nc",
)
combined.band.values

Then mask the cube with the combined band, exactly as in chapter 6:

In [ ]:
masked_cs = mask_stac_clouds(
    stac=BASE,
    cloud="../results/test_cloud_shadows.nc",
    mask_layer="cloudshadow_mask_70",
    output="../results/test_masked_cloudshadow.nc",
)